# 02 Build Embeddings

Reads `legal.bns_chunks`, computes sentence-transformers embeddings,
builds a FAISS index, and persists it to DBFS.

In [ ]:
%pip install -q tiktoken faiss-cpu pypdf python-dotenv gradio mlflow sentence-transformers llama-cpp-python
# dbutils.library.restartPython()

In [ ]:
import os
import sys
from pathlib import Path

REPO_PATH = "/Workspace/Repos/nyaya-bodh"
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.environ.setdefault("ARTIFACTS_DIR", "/dbfs/FileStore/nyaya_bodh")


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
chunks_pd = spark.table("legal.bns_chunks").select("section_id", "title", "chunk_id", "chunk_index", "text").toPandas()

print(f"Loaded {len(chunks_pd)} chunks")

In [ ]:
from config import CONFIG
from src.vector_store import VectorStore

chunks_pd.to_parquet(CONFIG.chunks_parquet_path, index=False)
store = VectorStore.build(chunks_pd)
store.save(CONFIG.faiss_index_path)
print(f"FAISS index written to {CONFIG.faiss_index_path}")

In [ ]:
import pandas as pd
schemes_pd = spark.table("legal.schemes").toPandas()
schemes_pd.to_parquet(CONFIG.schemes_parquet_path, index=False)

ipc_pd = spark.table("legal.ipc_bns_map").toPandas()
ipc_pd.to_parquet(CONFIG.ipc_bns_parquet_path, index=False)

print("Schemes and IPC mapping cached as parquet for the app runtime.")

In [ ]:
from src.pipeline import NyayaBodhPipeline

pipeline = NyayaBodhPipeline(store=store)
sample = pipeline.answer("What is the punishment for theft under BNS?")
print(sample.answer)
print("Citations:", [c.section_id for c in sample.citations])